Install and Import Required Libraries
Run this in your first cell to install any necessary scraping and machine learning libraries.

In [1]:
# Run this cell to install required libraries if not already installed
%pip install pandas scikit-learn requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\DELL\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.



Download Directly in Your Notebook

Once the environment variables are injected, the official Python library automatically authenticates without requiring a kaggle.json file. Run this script directly inside your OpenShift AI notebook cell:

In [ ]:
import os
import subprocess

# 1. Install the Kaggle CLI tool
subprocess.run(["pip", "install", "--quiet", "kaggle"])

# 2. Define the dataset slug (e.g., 50k IMDb reviews)
dataset_slug = "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
download_path = "./data"

# 3. Execute the secure download and unzip
os.makedirs(download_path, exist_ok=True)
subprocess.run([
    "kaggle", "datasets", "download", 
    "-d", dataset_slug, 
    "-p", download_path, 
    "--unzip"
])

print(f"Dataset downloaded successfully to: {download_path}")


Load Kaggle Dataset

In [11]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load the 50K IMDB dataset from Kaggle
# Make sure 'IMDB Dataset.csv' is in the same directory as your notebook
df_kaggle = pd.read_csv('./data/IMDB Dataset.csv')

# Convert text labels ('positive'/'negative') to binary (1/0)
df_kaggle['label'] = df_kaggle['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

print("Dataset shape:", df_kaggle.shape)
df_kaggle.head()

Dataset shape: (50000, 3)


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


Train the Sentiment Analysis Model
We train the model using the 50K dataset with TF-IDF vectorization and Logistic Regression.

In [12]:
X = df_kaggle['review']
y = df_kaggle['label']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorize text using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate model performance
preds = model.predict(X_test_vec)
print(f"Model Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")

Model Accuracy: 89.10%


Predict Movie Category (Good or Bad) Using an IMDb Review Comments
This function combines the scraper and your trained model to analyze an IMDb Review Comments and output an overall verdict.

In [ ]:
def classify_movie_from_uri(reviews, model, vectorizer):
    print(f"Analyzing reviews for the movie...")
    
    if not reviews:
        return "No reviews found or access was blocked by IMDb."
    
    # Vectorize and predict sentiment for each review
    vec_reviews = vectorizer.transform(reviews)
    predictions = model.predict(vec_reviews)
    
    # Calculate percentage of positive (good) reviews
    positive_count = sum(predictions)
    total_reviews = len(predictions)
    positive_ratio = (positive_count / total_reviews) * 100
    
    # Determine overall movie category
    verdict = "Good Movie 😊" if positive_ratio >= 50 else "Bad Movie 😞"
    
    print(f"\n--- Results ---")
    print(f"Total Reviews Analyzed: {total_reviews}")
    print(f"Positive Reviews: {positive_ratio:.1f}%")
    print(f"Overall Categorization: {verdict}\n")
    
    # Return sample breakdown DataFrame
    results_df = pd.DataFrame({
        "Review": reviews,
        "Prediction": ["Good" if p == 1 else "Bad" for p in predictions]
    })
    return results_df

# --- Example Usage ---
reviews = [
    "I absolutely loved this movie! The acting was superb and the plot was thrilling.",
    "This was a waste of time. The story was boring and predictable.",
    "An amazing experience! I would watch it again.",
    "Not worth the hype. I expected more from this film.",
    "A masterpiece! The cinematography and soundtrack were incredible."
]
result_df = classify_movie_from_uri(reviews, model, vectorizer)
print("result_df:")
print(result_df)

Fetching and analyzing reviews for: https://www.imdb.com/title/tt0111161/

--- Results ---
Total Reviews Analyzed: 3
Positive Reviews: 66.7%
Overall Categorization: Good Movie 😊

result_df:
                                              Review Prediction
0  It is no wonder that the film has such a high ...       Good
1  Don't get me wrong, Jenna Ortega's performance...       Good
2  It was watchable but I didn't love it.\n\nI'm ...        Bad
